# Day 3 · Occupancy Rate + Traffic Rotation
**Owner:** Sloane · Issue #29

Completes the Day 3 checklist:

1. Derive `occupancy_rate = occupied_rooms / available_rooms` for Occupancy
   (persisted via `sparkcityx.transforms.add_occupancy_rate` so it is reused
   rather than recomputed ad hoc)
2. Confirm the shared Day 3 aggregation code (`day3_stuff.txt`'s
   `prepare_correlation_dataset`) picks up `occupancy_rate` correctly
3. Rotate onto Traffic: add domain-specific anomaly flags
   (`sparkcityx.transforms.add_traffic_anomaly_flags`)

Run `uv run python scripts/generate-data.py --records 36000` first so
`data/raw/occupancy_data.csv` and `data/raw/traffic_sensors.csv` exist
locally before running this notebook.

## Setup

In [ ]:
import os

os.environ.setdefault(
    "JAVA_HOME",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
)

from pyspark.sql import SparkSession, functions as F

from sparkcityx.transforms import add_occupancy_rate, add_traffic_anomaly_flags

spark = (
    SparkSession.builder
    .appName("day3-occupancy-rotation")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 1. Derive `occupancy_rate`

`day3_stuff.txt`'s `prepare_correlation_dataset` references an
`occupancy_rate` column that does not exist in the raw generated data.
`add_occupancy_rate` derives it: `occupied_rooms / available_rooms`, with a
null (not a crash) when `available_rooms` is 0.

In [1]:
occupancy_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/raw/occupancy_data.csv")
)
occupancy_df = add_occupancy_rate(occupancy_df)

occupancy_df.select(
    "sensor_id", "timestamp", "available_rooms", "occupied_rooms", "occupancy_rate"
).show(5)

+---------+-------------------+---------------+--------------+-------------------+
|sensor_id|          timestamp|available_rooms|occupied_rooms|     occupancy_rate|
+---------+-------------------+---------------+--------------+-------------------+
| OCC-0001|2025-01-01 00:00:00|            235|           130| 0.5531914893617021|
| OCC-0002|2025-01-01 00:15:00|             28|            10|0.35714285714285715|
| OCC-0003|2025-01-01 00:30:00|           1227|           606| 0.4938875305623472|
| OCC-0004|2025-01-01 00:45:00|            153|            86| 0.5620915032679739|
| OCC-0005|2025-01-01 01:00:00|             42|            29| 0.6904761904761905|
+---------+-------------------+---------------+--------------+-------------------+
only showing top 5 rows


## 2. Confirm the shared Day 3 aggregation code picks it up

This mirrors `day3_stuff.txt`'s `prepare_correlation_dataset` occupancy
block exactly: hourly bucketing + `F.avg("occupancy_rate")`. Before this
column existed, this call would fail with an analysis exception.

In [2]:
occupancy_hourly = (
    occupancy_df
    .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
    .groupBy("hour_timestamp")
    .agg(
        F.avg("occupancy_rate").alias("avg_occupancy_rate"),
        F.avg("guests").alias("avg_guests"),
        F.count("*").alias("occupancy_readings"),
    )
)

occupancy_hourly.orderBy("hour_timestamp").show(5)
print(f"Total hourly buckets: {occupancy_hourly.count()}")

+-------------------+-------------------+----------+------------------+
|     hour_timestamp| avg_occupancy_rate|avg_guests|occupancy_readings|
+-------------------+-------------------+----------+------------------+
|2025-01-01 00:00:00|0.49157834508372006|    401.25|                 4|
|2025-01-01 01:00:00| 0.5138544690870273|     595.0|                 4|
|2025-01-01 02:00:00| 0.5637822898818283|    725.75|                 4|
|2025-01-01 03:00:00| 0.5983906580023699|    676.75|                 4|
|2025-01-01 04:00:00| 0.5129132080966943|     361.5|                 4|
+-------------------+-------------------+----------+------------------+
only showing top 5 rows
Total hourly buckets: 8999


### Findings — `occupancy_rate`

- The shared aggregation runs cleanly against real data: 8,999 hourly
  buckets, ~4 readings per bucket (matches four sensors on a 15-minute
  cadence).
- This unblocks the cross-sensor correlation analysis in
  `day3_stuff.txt`, and sets up the Day 3 cross-check (issue #33) once
  Hakeem's `expense_rate`/`revenue_rate` exist alongside this.

## 3. Rotate onto Traffic: domain-specific anomaly flags

Per the Day 3 rotation requirement, applying a meaningful transformation to
a dataset outside Occupancy. This flags physically impossible Traffic
readings (mirrors the domain rules stubbed in `day2_stuff.txt`'s
`detect_domain_outliers` for traffic): `avg_speed` outside 0-120 km/h, or a
negative `vehicle_count`.

In [3]:
traffic_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/raw/traffic_sensors.csv")
)
traffic_flagged = add_traffic_anomaly_flags(traffic_df)

traffic_flagged.groupBy(
    "speed_outlier_domain", "vehicle_count_outlier_domain"
).count().show()

+--------------------+----------------------------+-----+
|speed_outlier_domain|vehicle_count_outlier_domain|count|
+--------------------+----------------------------+-----+
|               false|                       false|36000|
+--------------------+----------------------------+-----+


### Findings — Traffic anomaly flags

- All 36,000 Traffic rows pass both domain checks — the generated data has
  no physically impossible speeds or vehicle counts.
- Same conclusion as the Day 2 Occupancy quality check: the synthetic data
  is clean by construction, so these flags currently find nothing, but the
  transformation is real, tested (`tests/test_transforms.py`), and ready to
  catch bad readings if the generator or a future real feed introduces
  them.

In [4]:
spark.stop()